# Chapter 12 · Whole school, mini-batch, one student
**Deck 2 · Live demo.** Train on 1,000 students three ways with the **same amount of work**, and see why every modern model uses mini-batches.

**You will:** build a noisy school · find the best possible line · write one training function for any batch size · prove the slide table.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
Each method may look at 50,000 students in total (η = 0.01). Whole school: 50 steps. Mini-batch of 32: 1,563 steps. One student: 50,000 steps. Which ends closest to w = 5, b = 40?

In [ ]:
my_prediction = 32   # ← batch size you think wins: 1000, 32 or 1

## 2 · The school
Same rule as our class (5 marks per hour + 40), but each real mark wobbles up to ±5. The numbers are generated by a fixed recipe, so the slides and this notebook see exactly the same students.

In [ ]:
N = 1000
i = np.arange(N)
hours = 1 + (i * 3) % 10                  # 1 … 10 hours
wobble = (i * 37) % 11 - 5                # −5 … +5 marks
marks = 5 * hours + 40 + wobble
order = (i * 7919) % N                    # a fixed shuffled order to visit students
print(hours[:10], marks[:10])

## 3 · The best any line can do
Because of the wobble no line hits every student. Least squares finds the best line directly; its loss is the floor.

In [ ]:
def loss(w, b):
    return np.mean((w * hours + b - marks) ** 2)

best_w, best_b = np.polyfit(hours, marks, 1)
print(f'best line: w = {best_w:.3f}, b = {best_b:.3f}, loss = {loss(best_w, best_b):.2f}')

## 3b · Is a sample enough?
The slope for w at the start (w = 0, b = 0), from all 1,000 students vs from groups of 32.

In [ ]:
def start_slope(idx):
    return 2 * np.mean((0 * hours[idx] + 0 - marks[idx]) * hours[idx])

all_students = start_slope(np.arange(N))
batches = [start_slope(order[k * 32:(k + 1) * 32]) for k in range(3)]
print(f'all 1,000 students: {all_students:.1f}')
print('three groups of 32:', [round(v, 1) for v in batches])
assert round(all_students, 1) == -825.0 and [round(v, 1) for v in batches] == [-819.8, -795.1, -849.1]
print('✓ 32 students give nearly the same slope, for 31× less work')

## 4 · One training function, any batch size
The update rule from Chapter 10, but each step only looks at `batch_size` students, taken in turn from the shuffled order.

$$w \leftarrow w - \eta \cdot \frac{2}{B}\sum_{\text{batch}} (\hat{y}-y)\,x \qquad b \leftarrow b - \eta \cdot \frac{2}{B}\sum_{\text{batch}} (\hat{y}-y)$$

In [ ]:
def train(batch_size, budget=50_000, eta=0.01):
    w = b = 0.0
    seen = pos = steps = 0
    curve = [(0, loss(w, b))]
    while seen < budget:
        idx = order[(pos + np.arange(batch_size)) % N]
        pos = (pos + batch_size) % N
        err = w * hours[idx] + b - marks[idx]
        w -= eta * 2 * np.mean(err * hours[idx])
        b -= eta * 2 * np.mean(err)
        seen += batch_size
        steps += 1
        curve.append((seen, loss(w, b)))
    return steps, w, b, loss(w, b), np.array(curve)

NAMES = {1000: 'Whole school (1,000)', 32: 'Mini-batch (32)', 1: 'One student (1)'}
results = {bs: train(bs) for bs in (1000, 32, 1)}
for bs, (steps, w, b, L, _) in results.items():
    print(f'{NAMES[bs]:<22} {steps:>6,} steps → w {w:.2f}, b {b:.2f}, loss {L:.1f}')

## 5 · PROOF
The table on the slide.

In [ ]:
r = {bs: tuple(round(v, 2) for v in res[1:4]) + (res[0],) for bs, res in results.items()}
assert r[1000] == (9.48, 8.82, 218.46, 50), r[1000]
assert r[32] == (5.03, 39.95, 10.02, 1563), r[32]
assert r[1] == (5.49, 39.94, 18.81, 50000), r[1]
assert round(loss(best_w, best_b), 2) == 10.01
assert my_prediction == 32, 'the mini-batch wins'
print('✓ proof: whole school 218.5 · mini-batch 10.0 (best possible 10.0) · one student 18.8')

## 6 · Picture

In [ ]:
for bs, color in [(1000, 'tab:blue'), (32, 'tab:green'), (1, 'tab:red')]:
    c = results[bs][4]
    plt.plot(c[:, 0], np.minimum(c[:, 1], 300), color=color, label=NAMES[bs])
plt.axhline(loss(best_w, best_b), ls='--', color='gray', label='best possible ≈ 10')
plt.xlabel('students looked at (work done)'); plt.ylabel('loss (capped at 300)'); plt.legend(); plt.show()

## 7 · Break it
Give the whole school 100× more work. Does it catch up?

In [ ]:
steps, w, b, L, _ = train(1000, budget=5_000_000)
print(f'{steps:,} steps → w {w:.2f}, b {b:.2f}, loss {L:.2f}  (it gets there, at 100× the cost)')

## 8 · Your turn
Try batch sizes 8 and 128 with the same 50,000 budget. Where is the sweet spot?

In [ ]:
for bs in (8, 32, 128):
    print(bs, [round(v, 2) for v in train(bs)[1:4]])

## Recap
- Whole school per step: exact slope, but very few steps for the work
- One student per step: many steps, but jumpy: never settles
- Mini-batch: the best of both, and the standard way to train every modern model
- One pass over all students = one epoch; 50,000 looks = 50 epochs

**Next (Chapter 13):** more clues than hours.